In [ ]:
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = Path.cwd().parent
PROJECT_DIR = BASE_DIR.resolve().parents[1]
DATA_DIR = BASE_DIR / "data"

all_chunks = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\nSection ", "\nSection ", "\n\n", "\n", " "]
)

for pdf_path in DATA_DIR.rglob("*.pdf"):
    category = pdf_path.parent.name      # e.g. "civil", "criminal", "technology"
    act_name = pdf_path.stem             # e.g. "ipc_1860", "bns_2023"
    
    # Read PDF using pypdf
    reader = PdfReader(str(pdf_path))
    pdf_docs = []
    
    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        # Skip blank pages
        if not text.strip():
            continue
            
        # Wrap extracted text and metadata into a LangChain Document
        pdf_docs.append(
            Document(
                page_content=text,
                metadata={
                    "category": category,
                    "act": act_name,
                    "source_file": pdf_path.name,
                    "page": idx + 1,
                }
            )
        )
    
    # Split documents into chunks (metadata is automatically propagated)
    if pdf_docs:
        doc_chunks = text_splitter.split_documents(pdf_docs)
        all_chunks.extend(doc_chunks)

# Store in Chroma with metadata preserved
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./legal_chroma_db"
)